## **1. Setup**
---

In [1]:
import os
import glob
import shutil
import tarfile
import random
import warnings
import kagglehub

import cv2
import numpy as np
from tqdm.notebook import tqdm

import torch
from facenet_pytorch import MTCNN

warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
def set_seed(seed, reproducibility=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Reproducibility or Performance
    torch.backends.cudnn.deterministic = reproducibility
    torch.backends.cudnn.benchmark     = not reproducibility

SEED = 42
set_seed(SEED, reproducibility=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
nGPU   = torch.cuda.device_count()
print(f"Device: {DEVICE}")
print(f"nGPUs : {nGPU}")

Device: cuda
nGPUs : 1


In [3]:
face_detector = MTCNN(image_size=112, margin=10, device='cuda:0', post_process=False)

In [4]:
DATASET_DIR = 'data/dataset'
TRAIN_DIR   = 'data/train'
TEST_DIR    = 'data/test'
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

## **2. Download Data**
---

In [5]:
def download_kagglehub_dataset(ds_name):
    path = kagglehub.dataset_download(ds_name)
    print(f"Org path: {path}")
    print(f"Dataset : {ds_name}")
    print(f"Contains: {os.listdir(path)}")
    
    dst = os.path.join(DATASET_DIR, ds_name)
    os.makedirs(dst, exist_ok=True)
    
    for item in os.listdir(path):
        s = os.path.join(path, item)
        d = os.path.join(dst, item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        elif os.path.isfile(s):
            shutil.copy2(s, d)

### 2.1. Training Data

In [6]:
download_kagglehub_dataset(ds_name='hereisburak/pins-face-recognition')

Org path: C:\Users\nguye\.cache\kagglehub\datasets\hereisburak\pins-face-recognition\versions\1
Dataset : hereisburak/pins-face-recognition
Contains: ['105_classes_pins_dataset']


In [7]:
download_kagglehub_dataset(ds_name='stoicstatic/face-recognition-dataset')

Org path: C:\Users\nguye\.cache\kagglehub\datasets\stoicstatic\face-recognition-dataset\versions\10
Dataset : stoicstatic/face-recognition-dataset
Contains: ['Extracted Faces', 'Face Data']


### 2.2. Testing Data

In [8]:
download_kagglehub_dataset(ds_name='atulanandjha/lfwpeople')

Org path: C:\Users\nguye\.cache\kagglehub\datasets\atulanandjha\lfwpeople\versions\3
Dataset : atulanandjha/lfwpeople
Contains: ['lfw-funneled.tgz', 'pairs.txt', 'pairsDevTest.txt', 'pairsDevTrain.txt']


In [10]:
tgz_path = 'data/dataset/atulanandjha/lfwpeople/lfw-funneled.tgz'

with tarfile.open(tgz_path, 'r:gz') as tar:
    tar.extractall(path=TEST_DIR)
os.rename('data/test/lfw_funneled', 'data/test/LFW')

## **3. Prepare Data**
---

In [11]:
def format_data(data_dir, target_dir):
    os.makedirs(target_dir, exist_ok=True)
    
    person_dirs = sorted([d for d in glob.glob(os.path.join(data_dir, '*')) 
                          if os.path.isdir(d)])
    
    person2id = {os.path.basename(p): str(i) for i, p in enumerate(person_dirs)}

    for pname, pid in tqdm(person2id.items(), desc=f'Formatting {os.path.basename(target_dir)} data...'):
        src_dir = os.path.join(data_dir, pname)
        dst_dir = os.path.join(target_dir, pid)
        os.makedirs(dst_dir, exist_ok=True)

        filenames = [f for f in os.listdir(src_dir) if os.path.isfile(os.path.join(src_dir, f))]
        for i, filename in enumerate(filenames):
            src_file = os.path.join(src_dir, filename)
            dst_file = os.path.join(dst_dir, f"{pid}_{i:03d}.jpg")
            shutil.copyfile(src_file, dst_file)
            
        # counter = 0
        # for filename in filenames:
        #     src_file = os.path.join(src_dir, filename)

        #     try:
        #         img  = Image.open(src_file).convert('RGB')
        #         face = face_detector(img)
        #         if face is not None:
        #             aligned_face = transforms.ToPILImage()(face.cpu())
        #             dst_file     = os.path.join(dst_dir, f'{pid}_{counter:03d}.jpg')
        #             aligned_face.save(dst_file)
        #             counter += 1
        #     except Exception as e:
        #         continue

In [12]:
def dataset_info(dataset_path):
    print(f"Dataset: {os.path.basename(dataset_path)}")
    print(f"Length : {len([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])} (folder)")
    print(f"Files  : {[f for f in os.listdir(dataset_path) if os.path.isfile(os.path.join(dataset_path, f))]}")
    
    img_shape = set()
    img_cnt   = ppl_cnt = 0
    for person_id in tqdm(os.listdir(dataset_path), desc=f"Counting {os.path.basename(dataset_path)}...", leave=False):
        person_path = os.path.join(dataset_path, person_id)
        
        if os.path.isdir(person_path):
            ppl_cnt += 1
            
            for img_name in os.listdir(person_path):
                img_path = os.path.join(person_path, img_name)
                
                if os.path.isfile(img_path):
                    img = cv2.imread(img_path)
                    
                    if img is not None:
                        img_shape.add(img.shape)
                        img_cnt += 1
                        
    print(
        f"Total images: {img_cnt}\n"
        f"Total people: {ppl_cnt}\n"
        f"Image shape : {list(img_shape)[:5]}{'...' if len(img_shape) >= 5 else ''}\n"
    )

In [14]:
TRAIN_DATA1 = os.path.join(TRAIN_DIR, 'dataset1_pins_105')
TEST_DATA  = os.path.join(TEST_DIR, 'LFW')

format_data(
    data_dir='data/dataset/hereisburak/pins-face-recognition/105_classes_pins_dataset', 
    target_dir=TRAIN_DATA1
)

dataset_info(TRAIN_DATA1)
dataset_info(TEST_DATA)

Formatting dataset1_pins_105 data...:   0%|          | 0/105 [00:00<?, ?it/s]

Dataset: dataset1_pins_105
Length : 105 (folder)
Files  : []


Counting dataset1_pins_105...:   0%|          | 0/105 [00:00<?, ?it/s]

Total images: 17534
Total people: 105
Image shape : [(507, 510, 3), (466, 489, 3), (387, 315, 3), (463, 343, 3), (387, 326, 3)]...

Dataset: LFW
Length : 5749 (folder)
Files  : ['pairs.txt', 'pairs_01.txt', 'pairs_02.txt', 'pairs_03.txt', 'pairs_04.txt', 'pairs_05.txt', 'pairs_06.txt', 'pairs_07.txt', 'pairs_08.txt', 'pairs_09.txt', 'pairs_10.txt']


Counting LFW...:   0%|          | 0/5760 [00:00<?, ?it/s]

Total images: 13233
Total people: 5749
Image shape : [(250, 250, 3)]

